# 01 — Exploratory Data Analysis

Explores both datasets that feed into the multimodal fusion system:
the facial image dataset and the eye-tracking recordings.

## What this notebook does
- Analyses class distribution and image properties of the facial dataset
- Loads and inspects all 25 eye-tracking CSV files (57 participants)
- Engineers per-participant summary features from 1.3M raw gaze rows
- Compares ASD vs TD distributions across key gaze features
- Saves the engineered feature matrix for use in `03_eye_tracking.ipynb`

## Output
`eye_tracking_features.csv` — 57 × 38 feature matrix with class labels

In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import cv2
import random
import warnings
import glob

warnings.filterwarnings('ignore')

In [63]:
# Mount drive and set paths - run at start of every session
from google.colab import drive
drive.mount('/content/drive')

In [18]:
# Paths
BASE_PATH = '/content/drive/MyDrive/Dataset'
FACE_PATH = f'{BASE_PATH}/autism_dataset/images'
EYE_PATH = f'{BASE_PATH}/eye_tracking_dataset/autism_eye_data/Eye-tracking Output'

print(f"Face data path exists: {os.path.exists(FACE_PATH)}")
print(f"Eye tracking path exists: {os.path.exists(EYE_PATH)}")

## Face EDA

In [4]:
# Count images per split
for split in ['train', 'val', 'test']:
    split_path = f'{FACE_PATH}/{split}'
    images = [f for f in os.listdir(split_path) if f.endswith('.jpg')]
    labels = [f for f in os.listdir(split_path) if f.endswith('.txt')]
    print(f"{split}: {len(images)} images, {len(labels)} labels")

In [ ]:
# Class distribution
def get_class_distribution(split):
    split_path = f'{FACE_PATH}/{split}'
    autistic, non_autistic = 0, 0
    for f in os.listdir(split_path):
        if f.endswith('.jpg'):
            if 'Autsim' in f and 'No_' not in f:
                autistic += 1
            else:
                non_autistic += 1
    return autistic, non_autistic

results = {}
for split in ['train', 'val', 'test']:
    a, n = get_class_distribution(split)
    results[split] = {'Autistic': a, 'Non-Autistic': n}
    print(f"{split} — Autistic: {a}, Non-Autistic: {n}, Total: {a+n}")



In [6]:
# Plot
df_dist = pd.DataFrame(results).T
df_dist.plot(kind='bar', figsize=(8,5), color=['#E74C3C', '#2ECC71'])
plt.title('Class Distribution Across Splits')
plt.xlabel('Split')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

- There is a class imbalance for val and test!

In [7]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Sample Facial Images — Autistic vs Non-Autistic', fontsize=14)

autistic_imgs = [f for f in os.listdir(f'{FACE_PATH}/train') 
                 if f.endswith('.jpg') and 'No_' not in f][:4]
non_autistic_imgs = [f for f in os.listdir(f'{FACE_PATH}/train') 
                     if f.endswith('.jpg') and 'No_' in f][:4]

for i, img_name in enumerate(autistic_imgs):
    img = Image.open(f'{FACE_PATH}/train/{img_name}')
    axes[0, i].imshow(img)
    axes[0, i].set_title('Autistic', color='red', fontsize=10)
    axes[0, i].axis('off')

for i, img_name in enumerate(non_autistic_imgs):
    img = Image.open(f'{FACE_PATH}/train/{img_name}')
    axes[1, i].imshow(img)
    axes[1, i].set_title('Non-Autistic', color='green', fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150)
plt.show()

In [10]:
# Sample 50 images to check dimensions and aspect ratios
sizes = []
all_imgs = []
for split in ['train', 'val', 'test']:
    for f in os.listdir(f'{FACE_PATH}/{split}'):
        if f.endswith('.jpg'):
            all_imgs.append(f'{FACE_PATH}/{split}/{f}')

for img_path in random.sample(all_imgs, 50):
    img = Image.open(img_path)
    sizes.append(img.size)

sizes_df = pd.DataFrame(sizes, columns=['width', 'height'])
print(sizes_df.describe())
print(f"Most common size: {sizes_df.value_counts().index[0]}")

In [12]:
# Sample N label files to check class distribution
SAMPLE_SIZE = 50  # adjust as needed

class_counts = {0: 0, 1: 0, 2: 0}  # based on classes.txt
all_label_files = []

# Collect all label files first
for split in ['train', 'val', 'test']:
    split_path = f'{FACE_PATH}/{split}'
    for f in os.listdir(split_path):
        if f.endswith('.txt'):
            all_label_files.append(os.path.join(split_path, f))

# Randomly sample label files
sampled_files = random.sample(
    all_label_files,
    min(SAMPLE_SIZE, len(all_label_files))
)

# Count class labels from sampled files
for label_path in sampled_files:
    with open(label_path) as file:
        for line in file:
            cls = int(line.strip().split()[0])
            class_counts[cls] = class_counts.get(cls, 0) + 1

print(f"Class label distribution from {len(sampled_files)} sampled YOLO files:")
for cls, count in class_counts.items():
    print(f"  Class {cls}: {count} instances")

## EDA Summary — Facial Dataset

- Total images: 3,398 (train: 2,462 | val: 463 | test: 473)
- Image size: ~416x416px, consistent
- Classes: 0=autistic_face, 1=autism → merge to ASD; 2=no_autism → non-ASD
- Class imbalance: val/test skewed ~1:2 (autistic:non-autistic)
- Action: Apply weighted loss during training to handle imbalance

## Eye Tracking EDA

In [19]:
# Load all CSVs
eye_files = glob.glob(f'{EYE_PATH}/*.csv')

# Data overview
sample_df = pd.read_csv(eye_files[0])
print(f"\nSample file: {Path(eye_files[0]).name}")
print(f"Shape: {sample_df.shape}")
print(f"\nColumns ({len(sample_df.columns)}):")
print(sample_df.columns.tolist())

In [20]:
# Peek at one file
sample_df.head()

In [21]:
# Load all 25 CSVs into one dataframe, adding filename as participant reference
dfs = []
for f in sorted(glob.glob(f'{EYE_PATH}/*.csv')):
    df = pd.read_csv(f)
    df['file_id'] = Path(f).stem  # adds '1', '2', ... '25'
    dfs.append(df)

eye_df = pd.concat(dfs, ignore_index=True)

print(f"Total rows: {eye_df.shape[0]:,}")
print(f"Total columns: {eye_df.shape[1]}")
print(f"\nUnique participants (from 'Participant' col): {eye_df['Participant'].nunique()}")
print(f"Unique file IDs: {eye_df['file_id'].nunique()}")
print(f"\nRows per file:")
print(eye_df.groupby('file_id').size().sort_values())

In [22]:
# Checking for missing values
print("Missing values per column:")
missing = eye_df.isnull().sum()
missing_pct = (missing / len(eye_df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
print(missing_df[missing_df['missing_count'] > 0].sort_values('missing_%', ascending=False))

In [23]:
# Check if there's any ASD label in the data
for col in ['Category Group', 'Category Right', 'Category Left', 'Color', 'Stimulus', 'Content']:
    print(f"\n--- {col} ---")
    print(eye_df[col].value_counts().head(10))

In [24]:
# Understandint he metadata file
META_PATH = f'{BASE_PATH}/eye_tracking_dataset/autism_eye_data'
meta_files = glob.glob(f'{META_PATH}/*.csv')
print(f"Metadata file: {meta_files}")

In [26]:
meta_df = pd.read_csv(meta_files[0])
print(f"Shape: {meta_df.shape}")
print(f"\nColumns: {meta_df.columns.tolist()}")

In [27]:
meta_df.head(10)

In [42]:
# Check what participant IDs look like in eye tracking data
print("Eye tracking - sample Participant values:")
print(eye_df['Participant'].value_counts().head(20))

print(f"\nUnique participants in eye_df: {eye_df['Participant'].nunique()}")

# Check metadata participant IDs
print(f"\nMetadata ParticipantID sample:")
print(meta_df['ParticipantID'].tolist())

print(f"\nClass distribution in metadata:")
print(meta_df['Class'].value_counts())

In [47]:
# How many numbered participants match metadata?
meta_ids = set(meta_df['ParticipantID'].astype(str))
eye_participants = eye_df['Participant'].astype(str).unique()

# Filter out Unidentified
real_eye_participants = set([p for p in eye_participants if 'Unidentified' not in str(p)])

print(f"Participants in metadata: {len(meta_ids)} → {sorted(meta_ids)}")
print(f"\nReal numbered participants in eye data: {len(real_eye_participants)} → {sorted(real_eye_participants, key=lambda x: int(x))}")
print(f"\nParticipants in BOTH (can be matched): {len(meta_ids & real_eye_participants)}")
print(f"In metadata but NOT in eye data: {meta_ids - real_eye_participants}")
print(f"In eye data but NOT in metadata: {real_eye_participants - meta_ids}")

In [49]:
print("CARS Score missing values:", meta_df['CARS Score'].isnull().sum())
print(f"\nCARS Score stats:")
print(meta_df['CARS Score'].describe())

print(f"\nClass vs CARS Score:")
print(meta_df.groupby('Class')['CARS Score'].describe())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

meta_df['Class'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71'])
axes[0].set_title('Class Distribution (ASD vs TD)')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')

meta_df.boxplot(column='CARS Score', by='Class', ax=axes[1])
axes[1].set_title('CARS Score by Class')

plt.tight_layout()
plt.savefig('eye_tracking_labels.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFull metadata with usable participants only:")
usable_meta = meta_df[meta_df['ParticipantID'].astype(str).isin(real_eye_participants)]
print(f"Shape: {usable_meta.shape}")
print(usable_meta['Class'].value_counts())

In [53]:
eye_df.info()

In [52]:
# Key numeric columns we'll use for feature engineering
numeric_cols = [
    'Pupil Diameter Right [mm]', 'Pupil Diameter Left [mm]',
    'Point of Regard Right X [px]', 'Point of Regard Right Y [px]',
    'Point of Regard Left X [px]', 'Point of Regard Left Y [px]',
    'Gaze Vector Right X', 'Gaze Vector Right Y', 'Gaze Vector Right Z',
    'Gaze Vector Left X', 'Gaze Vector Left Y', 'Gaze Vector Left Z'
]

# Filter to real participants only (drop Unidentified)
eye_clean = eye_df[~eye_df['Participant'].astype(str).str.contains('Unidentified')].copy()
eye_clean['Participant'] = eye_clean['Participant'].astype(str)

print(f"Rows after dropping Unidentified: {len(eye_clean):,}")
print(f"Participants remaining: {eye_clean['Participant'].nunique()}")

# Stats per numeric column
print("\nNumeric feature summary:")
print(eye_clean[numeric_cols].describe().round(3))

In [54]:
# Columns to convert to numeric
cols_to_numeric = [
    'Pupil Diameter Right [mm]', 'Pupil Diameter Left [mm]',
    'Point of Regard Right X [px]', 'Point of Regard Right Y [px]',
    'Point of Regard Left X [px]', 'Point of Regard Left Y [px]',
    'Gaze Vector Right X', 'Gaze Vector Right Y', 'Gaze Vector Right Z',
    'Gaze Vector Left X', 'Gaze Vector Left Y', 'Gaze Vector Left Z',
    'Pupil Size Right X [px]', 'Pupil Size Right Y [px]',
    'Pupil Size Left X [px]', 'Pupil Size Left Y [px]',
    'Eye Position Right X [mm]', 'Eye Position Right Y [mm]', 'Eye Position Right Z [mm]',
    'Eye Position Left X [mm]', 'Eye Position Left Y [mm]', 'Eye Position Left Z [mm]',
    'Pupil Position Right X [px]', 'Pupil Position Right Y [px]',
    'Pupil Position Left X [px]', 'Pupil Position Left Y [px]',
    'Tracking Ratio [%]'
]

for col in cols_to_numeric:
    eye_clean[col] = pd.to_numeric(eye_clean[col], errors='coerce')

print("Dtypes after conversion:")
print(eye_clean[cols_to_numeric].dtypes)
print(f"\nMissing % after conversion (top 15):")
missing_after = (eye_clean[cols_to_numeric].isnull().sum() / len(eye_clean) * 100).round(1)
print(missing_after.sort_values(ascending=False).head(15))

## FEATURE ENGINEERING — Per-participant aggregation

In [55]:
def engineer_eye_features(df):
    features = {}
    
    # Gaze event counts from Category
    for side in ['Right', 'Left']:
        col = f'Category {side}'
        counts = df[col].value_counts()
        total = len(df)
        for event in ['Fixation', 'Saccade', 'Blink']:
            features[f'{side.lower()}_{event.lower()}_count'] = counts.get(event, 0)
            features[f'{side.lower()}_{event.lower()}_rate'] = counts.get(event, 0) / total
    
    # Tracking ratio
    features['tracking_ratio_mean'] = df['Tracking Ratio [%]'].mean()
    features['tracking_ratio_std'] = df['Tracking Ratio [%]'].std()
    
    # Pupil diameter
    for side in ['Right', 'Left']:
        col = f'Pupil Diameter {side} [mm]'
        valid = df[col].dropna()
        valid = valid[valid > 0]  # 0 values = tracker lost pupil
        features[f'pupil_{side.lower()}_mean'] = valid.mean() if len(valid) > 0 else np.nan
        features[f'pupil_{side.lower()}_std'] = valid.std() if len(valid) > 0 else np.nan
        features[f'pupil_{side.lower()}_missing_rate'] = df[col].isnull().sum() / len(df)
    
    # Gaze position (Point of Regard) 
    for side in ['Right', 'Left']:
        for axis in ['X', 'Y']:
            col = f'Point of Regard {side} {axis} [px]'
            valid = df[col].dropna()
            features[f'gaze_{side.lower()}_{axis.lower()}_mean'] = valid.mean()
            features[f'gaze_{side.lower()}_{axis.lower()}_std'] = valid.std()
    
    # Eye position (3D head position)
    for side in ['Right', 'Left']:
        for axis in ['X', 'Y', 'Z']:
            col = f'Eye Position {side} {axis} [mm]'
            valid = df[col].dropna()
            features[f'eye_pos_{side.lower()}_{axis.lower()}_mean'] = valid.mean() if len(valid) > 0 else np.nan
    
    # Tracking loss rate (gaze aversion signal)
    features['gaze_loss_rate_right'] = df['Point of Regard Right X [px]'].isnull().sum() / len(df)
    features['gaze_loss_rate_left'] = df['Point of Regard Left X [px]'].isnull().sum() / len(df)
    
    return features

# Apply per participant
participant_features = []
for pid, group in eye_clean.groupby('Participant'):
    feats = engineer_eye_features(group)
    feats['ParticipantID'] = str(pid)
    participant_features.append(feats)

eye_features_df = pd.DataFrame(participant_features)
print(f"Feature matrix shape: {eye_features_df.shape}")
print(f"\nFeature columns ({len(eye_features_df.columns)-1} features):")
print([c for c in eye_features_df.columns if c != 'ParticipantID'])

In [56]:
eye_features_df.head()

In [57]:
# Merge with metadata labels
usable_meta['ParticipantID'] = usable_meta['ParticipantID'].astype(str)
eye_features_labeled = eye_features_df.merge(usable_meta[['ParticipantID', 'Class']], on='ParticipantID', how='inner')

print(f"Shape after merge: {eye_features_labeled.shape}")
print(f"\nClass distribution:")
print(eye_features_labeled['Class'].value_counts())

print(f"\nMissing values per feature:")
missing = eye_features_labeled.isnull().sum()
print(missing[missing > 0])

In [59]:
eye_features_labeled.tail()

In [60]:
# Compare ASD vs TD distributions for most informative features
key_features = [
    'tracking_ratio_mean', 'gaze_loss_rate_right', 'gaze_loss_rate_left',
    'right_fixation_rate', 'right_saccade_rate', 'right_blink_rate',
    'pupil_right_mean', 'pupil_left_mean',
    'gaze_right_x_std', 'gaze_right_y_std'
]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    asd_vals = eye_features_labeled[eye_features_labeled['Class']=='ASD'][feat].dropna()
    td_vals = eye_features_labeled[eye_features_labeled['Class']=='TD'][feat].dropna()
    
    axes[i].hist(asd_vals, alpha=0.6, color='#e74c3c', label='ASD', bins=15)
    axes[i].hist(td_vals, alpha=0.6, color='#2ecc71', label='TD', bins=15)
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=8)

plt.suptitle('ASD vs TD — Eye Tracking Feature Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('eye_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [62]:
# Save to your own Google Drive root instead
eye_features_labeled.to_csv('/content/drive/MyDrive/eye_tracking_features.csv', index=False)
print("Eye tracking features saved to MyDrive root.")

### Facial Image Dataset
- **Total images:** 3,398 (~416×416px, YOLO format)
- **Classes:** 0=autistic_face, 1=autism → merged to ASD; 2=no_autism → TD
- **Split:** train/val/test
- **Issue:** Class imbalance in val/test (~1:2 ratio) → will use weighted loss during training
- **Saved in:** `Dataset/autism_dataset/images/`

### Eye-Tracking Dataset
- **Participants:** 57 usable (out of 59 — participants 12 & 16 excluded, no recordings)
- **Class distribution:** 27 ASD, 30 TD — near balanced ✅
- **Raw data:** 25 CSVs in `Eye-tracking Output/`, 1.3M rows after cleaning
- **Metadata:** 59 rows, columns: ParticipantID, Gender, Age, Class, CARS Score
- **CARS Score:** Only available for ASD participants — not usable as feature (label leakage)
- **Engineered features:** 36 per-participant aggregated features
- **Feature matrix saved:** `/content/drive/MyDrive/eye_tracking_features.csv`

### Key Findings
| Feature | Observation |
|---|---|
| `gaze_loss_rate_left` | Strongest ASD vs TD separator |
| `tracking_ratio_mean` | ASD children tracked less consistently |
| `right_blink_rate` | ASD children concentrated near 0, long tail |
| `pupil_right/left_mean` | High overlap — weak discriminator alone |

### Known Issues
| Issue | Resolution |
|---|---|
| Participants 12 & 16 missing eye data | Excluded from eye-tracking branch |
| 1 participant missing all left eye features | Impute with median in preprocessing |
| Facial class imbalance in val/test | Weighted loss function |
